In [1]:
import importlib
import train
import evaluator
import numpy as np
import torch

# Reload for fresh edits
importlib.reload(train)
importlib.reload(evaluator)

from train import generate_data, train_model # , evaluate
from evaluator import EFTReweighter

In [2]:
# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wc_dim = 16
N = 10
theta1 = [0.0] * wc_dim
step = 0  # Use 8 for reco

mass_regions = ["0to700", "700to900", "900toInf"]
cross_sections = {"0to700": 65.09, "700to900": 8.295, "900toInf": 14.03}
directory = "/eos/purdue/store/user/lingqian/fullrun2_eft_minitrees"
struct_dir = "/eos/purdue/store/user/lingqian/fullrun2_eft_sc"
eras = ["2016preVFP","2016postVFP"]
channels = ["ee", "emu", "mumu"]

# Create reweighter
reweighter = EFTReweighter(
    directory_path=directory,
    eras=eras,
    channels=channels,
    mass_regions=mass_regions,
    cross_sections=cross_sections,
    struct_const_dir=struct_dir,
    step=step
)



[INFO] 0to700 (2016preVFP): σ=65.090 pb, L=19500 pb⁻¹, Σw=4.302e+07 → scale=2.950e-02
[INFO] 0to700 (2016postVFP): σ=65.090 pb, L=16810 pb⁻¹, Σw=4.174e+07 → scale=2.621e-02
[INFO] 700to900 (2016preVFP): σ=8.295 pb, L=19500 pb⁻¹, Σw=1.187e+07 → scale=1.363e-02
[INFO] 700to900 (2016postVFP): σ=8.295 pb, L=16810 pb⁻¹, Σw=1.201e+07 → scale=1.161e-02
[INFO] 900toInf (2016preVFP): σ=14.030 pb, L=19500 pb⁻¹, Σw=1.774e+07 → scale=1.542e-02
[INFO] 900toInf (2016postVFP): σ=14.030 pb, L=16810 pb⁻¹, Σw=1.767e+07 → scale=1.335e-02


In [3]:
reweighter.load_structure_constants()
reweighter.load_observables()


Loading SC step0: 100%|██████████| 360/360 [01:07<00:00,  5.37it/s]


→ Loaded 360 SC files, combined shape=(21430321, 153)


Loading observables: 100%|██████████| 360/360 [05:46<00:00,  1.04file/s]


[INFO] Total events before selection: 21430321
[INFO] Total events after selection:  21430321
[INFO] Kept fraction: 100.00%


In [4]:
def load_powheg_observables(file_paths_powheg, obs_name, xsec_powheg, step="step0"):
    import numpy as np
    import awkward as ak
    import uproot

    obs_all = []
    weights_all = []
    total_events = 0

    # Count total POWHEG events
    for path in file_paths_powheg:
        with uproot.open(path) as file:
            total_events += file[f"ttBar_treeVariables_{step}"].num_entries

    for path in file_paths_powheg:
        with uproot.open(path) as file:
            tree = file[f"ttBar_treeVariables_{step}"]
            branches = tree.keys()

            # Always include gen leptons for masking
            base_branches = ["gen_l_pt", "gen_lbar_pt"]

            # Decide what additional branches to load
            if obs_name in branches:
                needed = base_branches + [obs_name]
            elif obs_name in ["gen_delta_y", "gen_tanh_delta_y"]:
                needed = base_branches + ["gen_top_rapidity", "gen_tbar_rapidity"]
            else:
                print(f"[WARN] {obs_name} not found in {path} and not derived; skipping.")
                continue

            # Load the branches
            arrays = tree.arrays(needed)
            mask = (arrays["gen_l_pt"] > 0) & (arrays["gen_lbar_pt"] > 0)
            arrays = arrays[mask]

            # Compute or assign observable
            if obs_name in branches:
                obs = arrays[obs_name]
            elif obs_name == "gen_delta_y":
                obs = np.abs(arrays["gen_top_rapidity"]) - np.abs(arrays["gen_tbar_rapidity"])
            elif obs_name == "gen_tanh_delta_y":
                delta_y = np.abs(arrays["gen_top_rapidity"]) - np.abs(arrays["gen_tbar_rapidity"])
                obs = np.tanh(delta_y)
            else:
                print(f"[WARN] Could not compute {obs_name}.")
                continue

            # Append values
            obs_all.append(obs)
            weights_all.append(np.full(len(obs), xsec_powheg / total_events))

    # Combine across all files
    if not obs_all:
        raise ValueError(f"No valid data found for {obs_name}")

    obs_all = ak.to_numpy(ak.concatenate(obs_all))
    weights_all = ak.to_numpy(ak.concatenate(weights_all))
    return obs_all, weights_all


import mplhep

import numpy as np
import matplotlib.pyplot as plt
import mplhep
import awkward as ak

def compare_powheg_vs_ctg0(
    obs_powheg_dict,
    weights_powheg_dict,
    obs_dict_ctg0,
    label_powheg,
    label_ctg0,
    var_names,
    bins=20,
    density=True,
    extra_weights_ctg0=None,   # NEW optional argument
    outdir=None                # optional directory to save plots
):
    """
    Compare POWHEG vs SMEFT (reweighted to SM) observables.
    If extra_weights_ctg0 is provided, it rescales SMEFT events
    (e.g. to match POWHEG m_tt mixture).
    """
    colors = ['blue', 'orange']

    for var_name in var_names:
        # --- Load observables ---
        pow_obs = ak.to_numpy(obs_powheg_dict[var_name])
        eft_obs = ak.to_numpy(obs_dict_ctg0[var_name])

        # --- Build weights ---
        pow_w = ak.to_numpy(weights_powheg_dict[var_name])
        if extra_weights_ctg0 is not None:
            eft_w = np.array(extra_weights_ctg0[: len(eft_obs)], dtype=float)
        else:
            eft_w = np.ones_like(eft_obs)

        obs_dict = {label_powheg: pow_obs, label_ctg0: eft_obs}
        weights_dict = {label_powheg: pow_w, label_ctg0: eft_w}

        # --- Plot setup ---
        fig, (ax, ax_ratio) = plt.subplots(
            2, 1, figsize=(8, 8),
            gridspec_kw={"height_ratios": [3, 1]},
            sharex=True
        )

        # choose bins
        if var_name == "gen_ttbar_mass":
            bins_arr = np.linspace(300, 1500, 21)
        else:
            bins_arr = np.histogram_bin_edges(
                np.concatenate(list(obs_dict.values())), bins=bins
            )

        global_max = 0
        reference_counts = None
        reference_errors = None

        # --- loop over samples ---
        for i, label in enumerate(obs_dict):
            obs = obs_dict[label]
            w = weights_dict[label]

            counts, _ = np.histogram(obs, bins=bins_arr, weights=w, density=density)
            sumw, _ = np.histogram(obs, bins=bins_arr, weights=w)
            sumw2, _ = np.histogram(obs, bins=bins_arr, weights=w**2)
            bin_widths = np.diff(bins_arr)
            errors = np.sqrt(sumw2) / (sumw * bin_widths)
            errors = np.nan_to_num(errors, nan=0.0, posinf=0.0, neginf=0.0)
            bin_centers = 0.5 * (bins_arr[:-1] + bins_arr[1:])
            global_max = max(global_max, np.max(counts + errors))

            ax.step(bins_arr, np.append(counts, counts[-1]),
                    where='post', color=colors[i % len(colors)],
                    linewidth=2, label=label)
            ax.errorbar(bin_centers, counts, yerr=errors,
                        fmt='o', color=colors[i % len(colors)],
                        markersize=4, capsize=2)

            # ratio vs POWHEG
            if i == 0:
                reference_counts = counts
                reference_errors = errors
            else:
                ratio = np.divide(
                    counts, reference_counts,
                    out=np.zeros_like(counts),
                    where=reference_counts != 0
                )
                rel_err_ref = np.divide(
                    reference_errors, reference_counts,
                    out=np.zeros_like(reference_errors),
                    where=reference_counts != 0
                )
                rel_err = np.divide(
                    errors, counts,
                    out=np.zeros_like(errors),
                    where=counts != 0
                )
                ratio_err = ratio * np.sqrt(rel_err**2 + rel_err_ref**2)
                ax_ratio.step(bins_arr, np.append(ratio, ratio[-1]),
                              where='post', color=colors[i % len(colors)])
                ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_err,
                                  fmt='o', color=colors[i % len(colors)],
                                  markersize=4, capsize=2)

        # --- styling ---
        ax.set_ylabel("Normalized Events" if density else "Events")
        ax.set_ylim(0, 1.1 * global_max)
        ax.legend(loc='best')
        ax.set_title(f"Distribution of {var_name}")
        mplhep.cms.label("Work in progress", data=True, ax=ax, loc=0)

        ax_ratio.set_xlabel(var_name)
        ax_ratio.set_ylabel("Ratio")
        ax_ratio.set_ylim(0.8, 1.2)
        ax_ratio.grid(True)

        plt.tight_layout()

        # --- save if requested ---
        if outdir is not None:
            import os
            os.makedirs(outdir, exist_ok=True)
            fig.savefig(os.path.join(outdir, f"{var_name}.pdf"))
            plt.close(fig)
        else:
            plt.show()


In [5]:
import os

#directory_path_powheg = '/eos/purdue/store/user/jthieman/'
directory_path_powheg='/depot/cms/users/jduarteq/'
# Common parameters
flavors = ['ee', 'emu', 'mumu']
indices = range(5)

# 2016preVFP files
desired_files_powheg = [
        f"2016preVFP/spinCorrInput_2016preVFP_August2025/Nominal/{flavor}/{flavor}_ttbarsignalplustau_fromDilepton_2016ULpreVFP_{i}.root"
        for flavor in flavors for i in indices
    ] + [
        f"2016postVFP/spinCorrInput_2016postVFP_August2025/Nominal/{flavor}/{flavor}_ttbarsignalplustau_fromDilepton_2016ULpostVFP_{i}.root"
        for flavor in flavors for i in indices
    ]
# Combine both VFP eras
#desired_files_powheg = desired_files_powheg_preVFP + desired_files_powheg_postVFP

# Full paths
file_paths_powheg = [os.path.join(directory_path_powheg, f) for f in desired_files_powheg]

# Cross section
xsec_powheg = 831.76 * 0.10706  # = 89.056 pb


In [6]:
import uproot

# pick the first file as example
file_path = file_paths_powheg[0]

# open the ROOT file
with uproot.open(file_path) as f:
    print("Keys in the file:", f.keys())  # shows all objects (trees, histograms, etc.)

    # let's assume the main TTree is called 'tree' or check first key
    tree_name = f.keys()[2]  # often the first key is the TTree
    tree = f[tree_name]
    
    print("Branches in the tree:", tree.keys())  # these are your "features"

Keys in the file: ['weightedEvents;1', 'ttBar_treeVariables_step8zWindow;1', 'ttBar_treeVariables_step7zWindow;1', 'ttBar_treeVariables_step0;1', 'ttBar_treeVariables_step7;1', 'ttBar_treeVariables_step8;1']
Branches in the tree: ['top_pt', 'top_phi', 'top_rapidity', 'top_eta', 'top_mass', 'tbar_pt', 'tbar_phi', 'tbar_rapidity', 'tbar_eta', 'tbar_mass', 'l_pt', 'l_eta', 'l_phi', 'l_mass', 'l_pdgid', 'lepton_index', 'lbar_pt', 'lbar_eta', 'lbar_phi', 'lbar_mass', 'lbar_pdgid', 'antilepton_index', 'b_pt', 'b_eta', 'b_phi', 'b_mass', 'bbar_pt', 'bbar_eta', 'bbar_phi', 'bbar_mass', 'nu_pt', 'nu_eta', 'nu_phi', 'nu_mass', 'nubar_pt', 'nubar_eta', 'nubar_phi', 'nubar_mass', 'met_pt', 'met_phi', 'met_mass', 'ttbar_pt', 'ttbar_phi', 'ttbar_rapidity', 'ttbar_eta', 'ttbar_delta_phi', 'ttbar_delta_eta', 'ttbar_delta_rapidity', 'ttbar_mass', 'llbar_pt', 'llbar_phi', 'llbar_rapidity', 'llbar_delta_phi', 'llbar_delta_eta', 'llbar_delta_rapidity', 'llbar_mass', 'MT2', 'all_mass', 'r_mass', 'x1', 'x2'

In [7]:
obs_powheg_dict = {}
weights_powheg_dict = {}

VARS =[
    'gen_ttbar_mass',
    'gen_b1k','gen_b2k','gen_b1r','gen_b2r','gen_b1n','gen_b2n',
    'gen_c_nn','gen_c_rr','gen_c_kk',
    'gen_ll_cHel','gen_ll_cLab', 'gen_llbar_delta_phi','gen_ttbar_rapidity',
    'gen_b1q', "gen_llbar_pt", "gen_l_eta"
] 

for obs_name in VARS:
    obs, weights = load_powheg_observables(file_paths_powheg, obs_name, xsec_powheg)
    obs_powheg_dict[obs_name] = obs
    weights_powheg_dict[obs_name] = weights


In [8]:
importlib.reload(evaluator)
import inspect

# Get the source code as a string
try:
    source_code = inspect.getsource(reweighter.get_final_weights)
    print(source_code)
except TypeError as e:
    print(f"Cannot get source: {e}") 

# Output:
# def greet(name):
#     print(f"Hello, {name}!")
#     return f"Completed greeting for {name}."


    def get_final_weights(self, wc_point):
        """Compute full EFT event weights, aligned with structure constants."""
        struct = self.structure_constants.cpu().numpy()
        n_struct = len(struct)
        n_obs = len(self.final_observables[next(iter(self.final_observables))])
        if n_struct != n_obs:
            n = min(n_struct, n_obs)
            print(f"[WARN] Structure constants ({n_struct}) and observables ({n_obs}) mismatch → truncating to {n}")
            struct = struct[:n]
            for key in list(self.final_observables.keys()):
                self.final_observables[key] = self.final_observables[key][:n]

        eft_w = Event_weight_prediction1.event_weights_lin_quad(struct, wc_point)[2] # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        base_w = self._base_weights()

        if len(base_w) != len(eft_w):
            n = min(len(base_w), len(eft_w))
            print(f"[WARN] Adjusting array lengths: base={len(base_w)} vs eft={len(eft

In [9]:
M = 15000000000


def evaluate_observables_at_wc(reweighter, wc_point, var_names, max_events=50000):
    observables = reweighter.resample_observables(wc_point, max_events=max_events)
    return {var: observables[var] for var in var_names if var in observables}
obs_ctg0 = evaluate_observables_at_wc(reweighter, wc_point=[0.0]*16, var_names=VARS, max_events=M)
print(VARS)
# --------------------------------------------------------
# Extend SMEFT observables (obs_ctg0) with derived variables
# --------------------------------------------------------
import numpy as np

def cal_gen_delta_y(events):
    """Δy = |y_t| - |y_tbar|"""
    return np.abs(events["gen_top_rapidity"]) - np.abs(events["gen_tbar_rapidity"])

def cal_gen_tanh_delta_y(events):
    """tanh(Δy) bounded between -1 and 1"""
    return np.tanh(cal_gen_delta_y(events))

# Compute derived observables only if base quantities exist
if "gen_top_rapidity" in obs_ctg0 and "gen_tbar_rapidity" in obs_ctg0:
    if "gen_delta_y" not in obs_ctg0:
        obs_ctg0["gen_delta_y"] = cal_gen_delta_y(obs_ctg0)
    if "gen_tanh_delta_y" not in obs_ctg0:
        obs_ctg0["gen_tanh_delta_y"] = cal_gen_tanh_delta_y(obs_ctg0)
else:
    print("[WARN] gen_top_rapidity or gen_tbar_rapidity missing in obs_ctg0 — cannot compute Δy or tanh(Δy).")


[INFO] Resampled 20983980 events (step=0) with EFT-weighted probabilities.
['gen_ttbar_mass', 'gen_b1k', 'gen_b2k', 'gen_b1r', 'gen_b2r', 'gen_b1n', 'gen_b2n', 'gen_c_nn', 'gen_c_rr', 'gen_c_kk', 'gen_ll_cHel', 'gen_ll_cLab', 'gen_llbar_delta_phi', 'gen_ttbar_rapidity', 'gen_b1q', 'gen_llbar_pt', 'gen_l_eta']
[WARN] gen_top_rapidity or gen_tbar_rapidity missing in obs_ctg0 — cannot compute Δy or tanh(Δy).


In [8]:
import matplotlib.pyplot as plt
import awkward as ak

VARS =[
    'gen_ttbar_mass',
    'gen_b1k','gen_b2k','gen_b1r','gen_b2r','gen_b1n','gen_b2n',
    'gen_c_nn','gen_c_rr','gen_c_kk',
    'gen_ll_cHel','gen_ll_cLab', 'gen_llbar_delta_phi','gen_ttbar_rapidity',
    'gen_crk_ckr', 'gen_crk_ckr_m', 'gen_cnr_crn',
    'gen_cnr_crn_m', 'gen_cnk_ckn', 'gen_cnk_ckn_m',
    'gen_b1q', "gen_llbar_pt", "gen_l_eta"
]  
#["gen_ttbar_mass", "gen_top_rapidity", "gen_tbar_rapidity"] #,"gen_delta_y","gen_tanh_delta_y"]

compare_powheg_vs_ctg0(
    obs_powheg_dict,
    weights_powheg_dict,
    obs_dict_ctg0=obs_ctg0,
    label_powheg="POWHEG",
    label_ctg0="EFT Reweighted to SM",
    var_names=VARS
)


NameError: name 'obs_ctg0' is not defined

In [17]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

VARS =[
    'gen_ttbar_mass',
    'gen_b1k','gen_b2k','gen_b1r','gen_b2r','gen_b1n','gen_b2n',
    'gen_c_nn','gen_c_rr','gen_c_kk',
    'gen_ll_cHel','gen_ll_cLab', 'gen_llbar_delta_phi','gen_ttbar_rapidity',
    'gen_crk_ckr', 'gen_crk_ckr_m', 'gen_cnr_crn',
    'gen_cnr_crn_m', 'gen_cnk_ckn', 'gen_cnk_ckn_m',
    'gen_b1q', "gen_llbar_pt", "gen_l_eta"
] 



# ==========================
# Evaluate at a specific WC
# ==========================
def evaluate_observables_at_wc(reweighter, wc_point, var_names, max_events=50000):
    observables = reweighter.resample_observables(wc_point, max_events=max_events)
    return {var: observables[var] for var in var_names if var in observables}

# ==========================
# Plot Comparison for 3 WC Points with Ratio
# ==========================

def compare_three_observables_with_ratio(obs_dict_0, obs_dict_1, #obs_dict_2,
                                         label_0, label_1, #label_2,
                                         bins=50, density=True):
    for var_name in obs_dict_0.keys():
        # Convert awkward arrays to NumPy
        data0 = np.asarray(obs_dict_0[var_name])
        data1 = np.asarray(obs_dict_1[var_name])
        #data2 = np.asarray(obs_dict_2[var_name])

        # Special binning for gen_ttbar_mass
        if var_name == "gen_ttbar_mass":
            bin_edges = np.linspace(350, 1500, bins+1)  # 50 bins from 300 to 1500
        else:
            combined = np.concatenate([data0, data1, data2])
            bin_edges = np.linspace(np.min(combined), np.max(combined), bins + 1)

        # Histogram counts
        counts0, _ = np.histogram(data0, bins=bin_edges, density=density)
        counts1, _ = np.histogram(data1, bins=bin_edges, density=density)
        #counts2, _ = np.histogram(data2, bins=bin_edges, density=density)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # Plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8),
                                       gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

        # Top: Distributions
        # Smooth counts
        sigma = 1.5  # adjust smoothing strength
        smooth0 = gaussian_filter1d(counts0, sigma)
        smooth1 = gaussian_filter1d(counts1, sigma)
        
        # Plot smooth curves
        ax1.plot(bin_centers, smooth0, label=label_0, linewidth=2)
        ax1.plot(bin_centers, smooth1, label=label_1, linewidth=2)
        #ax1.hist(bin_centers, bins=bin_edges, weights=counts2, histtype='step', label=label_2, linewidth=2)
        ax1.set_ylabel("Normalized Entries" if density else "Entries")
        ax1.set_title(f"Distribution of {var_name}")
        ax1.legend()
        ax1.grid(True)

        # Bottom: Ratios
        counts0_safe = np.where(counts0 > 0, counts0, np.nan)
        raw_ratio = counts1 / counts0_safe
        
        # Smooth the ratio directly
        ratio1 = gaussian_filter1d(raw_ratio, sigma)
        #ratio2 = counts2 / counts0_safe
        ax2.plot(bin_centers, ratio1, label=f"{label_1} / {label_0}", linewidth=2)
#ax2.plot(bin_centers, ratio2, drawstyle='steps-mid', label=f"{label_2} / {label_0}")
        ax2.set_xlabel(var_name)
        ax2.set_ylabel("Ratio")
        ax2.axhline(1.0, color='gray', linestyle='--')
        ax2.legend()
        #ax2.grid(True)

        # Apply log y-axis and x-limit only for gen_ttbar_mass
        if var_name == "gen_ttbar_mass":
            ax1.set_yscale('log')
            ax1.set_xlim(300, 1500)
            ax2.set_xlim(300, 1500)

        plt.tight_layout()
        plt.show()


# ==========================
# Configuration and Evaluation
# ==========================
M = 15000000
wc_dim = 16

# WC names (optional mapping)
wc_name = ['ctGRe', 'ctGIm', 'cQj18', 'cQj38', 'cQj11', 'cQj31', 'ctu8', 'ctd8',
           'ctj8', 'cQu8', 'cQd8', 'ctu1', 'ctd1', 'ctj1', 'cQu1', 'cQd1']

# SM point (ctGRe = 0)
theta_0 = [0.0] * wc_dim
obs_ctg0 = evaluate_observables_at_wc(
    reweighter=reweighter,
    wc_point=theta_0,
    var_names=VARS,
    max_events=M
)

# EFT point 1: ctGRe = 2
theta_2 = [0.0] * wc_dim
theta_2[6] = 2.0
obs_ctg2 = evaluate_observables_at_wc(
    reweighter=reweighter,
    wc_point=theta_2,
    var_names=VARS,
    max_events=M
)
"""
# EFT point 2: ctu8 = 10
theta_3 = [0.0] * wc_dim
theta_3[6] = 15.0
obs_ctg3 = evaluate_observables_at_wc(
    reweighter=reweighter,
    wc_point=theta_3,
    var_names=VARS,
    max_events=M
)
"""
#EFT point 3: ctGRe = 2, ctj8 = -2
theta_mixed = [0.0] * wc_dim
theta_mixed[0] = 2    # ctGRe
theta_mixed[6] = 2   # ctj8
obs_mixed = evaluate_observables_at_wc(
    reweighter=reweighter,
    wc_point=theta_mixed,
    var_names=VARS,
    max_events=M
)

# ==========================
# Plot all three with ratio
# ==========================
compare_three_observables_with_ratio(
    obs_dict_0=obs_ctg0,
    obs_dict_1=obs_ctg2,
    obs_dict_2=obs_mixed,
    label_0="SM",
    label_1="ctGRe = 10",
    label_2="ctGRe = 2, ctj8 = -2"
)


[INFO] Resampled 15000000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 15000000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 15000000 events (step=0) with EFT-weighted probabilities.


TypeError: compare_three_observables_with_ratio() got an unexpected keyword argument 'obs_dict_2'

In [19]:
import pickle
import gzip
import pandas as pd

def save_compressed_pickle(data: dict, filename: str):
    df = pd.DataFrame(data)
    filepath = f"{filename}.pkl.gz"
    with gzip.open(filepath, "wb") as f:
        pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Saved compressed pickle to {filepath}")

save_compressed_pickle(obs_mixed,"/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctu8_ctGRe/events_0")
#save_compressed_pickle(obs_ctg0,"/home/jpittard/depot/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_pickle/events_0")


Saved compressed pickle to /depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctu8_ctGRe/events_0.pkl.gz


In [54]:
observables = reweighter.resample_observables(theta_2, max_events=15000)
observables

[INFO] Resampled 15000 events (step=0) with EFT-weighted probabilities.


{'gen_top_pt': <Array [77.3, 271, 140, 193, ... 212, 106, 196] type='15000 * float32'>,
 'gen_top_phi': <Array [2.5, 0.478, -2.25, ... -0.805, -2.8] type='15000 * float32'>,
 'gen_top_rapidity': <Array [0.379, 1.08, -1.34, ... 0.606, -0.585] type='15000 * float32'>,
 'gen_top_eta': <Array [0.843, 1.22, -1.76, ... 1.03, -0.751] type='15000 * float32'>,
 'gen_top_mass': <Array [172, 172, 174, 171, ... 172, 171, 172] type='15000 * float32'>,
 'gen_tbar_pt': <Array [80.4, 219, 10.8, ... 178, 73.7, 173] type='15000 * float32'>,
 'gen_tbar_phi': <Array [-1.03, -2.65, -0.431, ... 2.07, 0.9] type='15000 * float32'>,
 'gen_tbar_rapidity': <Array [0.37, -1.08, 1.94, ... -1.23, -0.603] type='15000 * float32'>,
 'gen_tbar_eta': <Array [0.806, -1.27, 4.68, ... -2.08, -0.812] type='15000 * float32'>,
 'gen_tbar_mass': <Array [173, 170, 172, 173, ... 173, 172, 173] type='15000 * float32'>,
 'gen_l_pt': <Array [40.4, 148, 31.2, ... 36.3, 19.1, 125] type='15000 * float32'>,
 'gen_l_eta': <Array [0.36, 

In [ ]:
# Generate training data
X_all, Y_all = [], []

VARS =[
    'gen_ttbar_mass',
    'gen_ttbar_rapidity',
]
th0 = np.zeros(wc_dim, dtype=float)
print(th0)
#obs0 = reweighter.resample_observables(th0, max_events=M)
#print(obs0)

for X, Y, _ in generate_data(reweighter, VARS, wc_dim, N=N, M=M, theta1=theta1):
    X_all.append(X)
    print(X)
    Y_all.append(Y)

X_all = np.vstack(X_all)
Y_all = np.hstack(Y_all)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[INFO] Resampled 15000000 events (step=0) with EFT-weighted probabilities.


In [14]:
X_all

array([[ 4.40070892e+02,  8.91744435e-01,  8.61980207e-03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 6.88448303e+02,  1.57928407e-01, -1.98893949e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 5.58305847e+02,  8.99192035e-01, -3.32368106e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 4.22888092e+02,  3.97150874e-01, -1.38253555e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 3.96025726e+02, -8.44381034e-01, -6.70546517e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 4.41589355e+02,  9.11282897e-01,  2.67627481e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [ ]:
# Train model
print(len(VARS))
model, scaler, (X_test, Y_test) = train_model(X_all, Y_all,  VARS, input_dim=len(VARS))

10


In [ ]:
evaluate(model, scaler, X_test, Y_test, device=device)
